In [1]:

!pip install -q langchain langchain-groq

import os
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
if not api_key:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets first.")

os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Setup successful!")


Setup successful!


In [2]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS: Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
       Orders need 24 hours' notice. All cakes have eggless versions.
RULES: Never promise a delivery time or place.
       If it isn't in FACTS, say "I'll check with the owner."
       Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })
    history.extend([
        ("human", question),
        ("ai", answer)
    ])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))


Yes, we offer eggless versions of all our cakes—plum, chocolate, and cupcakes. Let me know what you'd like!
I'll check with the owner.


In [3]:

import csv

def run_tests(path="/content/tests.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += int(ok)

        print(
            "PASS" if ok else "FAIL",
            "|", row["question"],
            "→", answer[:100]
        )

    rate = passed / len(rows) if rows else 0
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")

run_tests()


PASS | What cakes do you sell? → We sell Plum cake (₹500), Chocolate cake (₹650), and Cupcakes (₹80 each). All have eggless versions.
PASS | How much is a plum cake? → A plum cake costs Rs 500. Let me know if you’d like to order or need more details!
PASS | What is the price of chocolate cake? → The chocolate cake costs Rs 650.
PASS | How much does one cupcake cost? → One cupcake costs Rs 80 each. Let me know if you’d like to order!
PASS | How much are 3 cupcakes? → Three cupcakes cost Rs 240. Please note we need a 24‑hour notice for orders. Let me know if you’d li
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes—plum, chocolate, and cupcakes alike.
FAIL | Can I order a cake for tomorrow? → Sure, we can take your order for tomorrow if you place it now. Please let me know which cake you'd l
PASS | Can you guarantee delivery at 5 PM? → I’m not able to confirm a specific delivery time. I’ll check with the owner and get back to you.
PASS | Do you sell don

In [6]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Do you make eggless cakes?"}
    ]
})

print(result["messages"][-1].content)


Yes, we offer egg‑less versions for all our cakes—plum, chocolate, and cupcakes. Let me know which you’d like, and we can discuss the order details. Looking forward to helping you!


In [7]:
#I choosen Coffee Shop using  AI Assistant using LangChain + Groq.

In [10]:
SYSTEM = """ROLE:
You are a friendly AI assistant for Coffee Corner, a local coffee shop.

FACTS:
Espresso price Rs 120.
Cappuccino price Rs 150.
Latte price Rs 170.
Cold coffee price Rs 180.
Bulk orders need 24 hours' notice.

RULES:
Never invent menu items or prices.
If information is unavailable, say "I'll check with the owner."
Calculate totals carefully.
Never promise a delivery time or place.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer.
"""


In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()
history = []

In [12]:
print(ask("What is the price of espresso?"))
print(ask("What is the price of cappuccino?"))
print(ask("Do you have cold coffee?"))

Hey! An espresso costs Rs 120. Let me know if you’d like to order one or anything else!
Cappuccino is Rs 150. Happy to help with your order!
Yes! Cold coffee is available at Rs 180. Let me know if you’d like to order one!


In [13]:
import os

print("Current folder:", os.getcwd())
print("CSV file exists:", os.path.exists("tests.csv"))

print("\\nActual tests.csv content:\\n")
with open("tests.csv", "r", encoding="utf-8") as file:
    print(file.read())

Current folder: /content
CSV file exists: True
\nActual tests.csv content:\n
question,must_include
What is the price of espresso?,120
How much is a cappuccino?,150
What is the price of latte?,170
How much does cold coffee cost?,180
What is the price of tea?,check with the owner
What is the total for 2 cappuccinos?,300



In [15]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        if ok:
            passed += 1
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:100])
    total = len(rows)
    print(f"\\n{passed}/{total} passed ({passed/total:.0%})")

run_tests()


PASS | What is the price of espresso? → The espresso costs ₹120.
PASS | How much is a cappuccino? → A cappuccino is Rs 150. Happy sipping!
PASS | What is the price of latte? → The latte costs Rs 170.
PASS | How much does cold coffee cost? → Cold coffee costs Rs 180. Let me know if you’d like to place an order!
PASS | What is the price of tea? → I’m sorry, we don’t have tea on our menu. I’ll check with the owner and get back to you.
PASS | What is the total for 2 cappuccinos? → For 2 cappuccinos, the total is Rs 300.
\n6/6 passed (100%)


In [22]:
!pip install -q langchain-openai

In [23]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

In [25]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=1.0
)
print("Temperature changed to 1.0!")

Temperature changed to 1.0!


In [26]:
chain = prompt | llm | StrOutputParser()

In [27]:
print("First Run:")
run_tests()

First Run:
PASS | What is the price of espresso? → The espresso costs Rs 120. Happy sipping!
PASS | How much is a cappuccino? → The cappuccino costs Rs 150.
PASS | What is the price of latte? → The latte costs Rs 170.
PASS | How much does cold coffee cost? → Cold coffee costs Rs 180. Feel free to let me know if you’d like to order!
PASS | What is the price of tea? → I’m not sure about tea. I’ll check with the owner.
PASS | What is the total for 2 cappuccinos? → Sure! Two cappuccinos cost Rs 300 in total. Happy sipping!
\n6/6 passed (100%)


In [28]:
print("Second Run:")
run_tests()

Second Run:
PASS | What is the price of espresso? → Sure! The espresso at Coffee Corner costs Rs 120. Let me know if you'd like to place an order.
PASS | How much is a cappuccino? → A cappuccino costs Rs 150.
PASS | What is the price of latte? → The latte is Rs 170. Enjoy!
PASS | How much does cold coffee cost? → Cold coffee costs Rs 180.
PASS | What is the price of tea? → I’ll check with the owner.
PASS | What is the total for 2 cappuccinos? → Hey! Two cappuccinos will cost Rs 300 in total. Enjoy!
\n6/6 passed (100%)


In [29]:
jobs = [
    {
        "job": "Calculate the total price of 3 cappuccinos",
        "type": "Plain Code"
    },
    {
        "job": "Answer customer questions about coffee drinks and prices",
        "type": "One LLM Call"
    },
    {
        "job": "Check coffee menu availability using a tool",
        "type": "Agent"
    }
]
for item in jobs:
    print("Job:", item["job"])
    print("Type:", item["type"])
    print()


Job: Calculate the total price of 3 cappuccinos
Type: Plain Code

Job: Answer customer questions about coffee drinks and prices
Type: One LLM Call

Job: Check coffee menu availability using a tool
Type: Agent

